# Example 5: Reasoning Mode（思考模式）

展示思考模式对输出质量的影响。

1.导入库与配置

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("HY3_API_KEY"),
    base_url="https://tokenhub.tencentmaas.com/v1",
)

2.思考模式对比

统一输入

In [2]:
# 思考模式对比
messages = [
    {"role": "user", "content": "小明有5个苹果，给了小红2个，又买了3个，最后还剩几个？"},
]

print("=== 思考模式对比 ===")
print("问题:", messages[0]["content"])
print()

=== 思考模式对比 ===
问题: 小明有5个苹果，给了小红2个，又买了3个，最后还剩几个？



关闭思考模式

In [3]:
print("1. 关闭思考模式（thinking: disabled）")
response = client.chat.completions.create(
    model="hy3",
    messages=messages,
    extra_body={"thinking": {"type": "disabled"}},
)

msg = response.choices[0].message
print("回答:", msg.content)
if hasattr(msg, "reasoning_content"):
    print("思考过程:", getattr(msg, "reasoning_content"))
else:
    print("思考过程: 无（思考模式已关闭）")
print(f"Token 消耗: {response.usage.total_tokens}")
print()

1. 关闭思考模式（thinking: disabled）
回答: 我们一步步算：

1. 小明一开始有 **5个苹果**  
2. 给了小红2个：  
   $5 - 2 = 3$ 个  
3. 又买了3个：  
   $3 + 3 = 6$ 个  

**最后还剩：6个苹果** ✅
思考过程: 无（思考模式已关闭）
Token 消耗: 107



开启思考模式

In [4]:
print("2. 开启思考模式（thinking: enabled）")
response = client.chat.completions.create(
    model="hy3",
    messages=messages,
    extra_body={"thinking": {"type": "enabled"}},
)

msg = response.choices[0].message
print("回答:", msg.content)
if hasattr(msg, "reasoning_content"):
    print("思考过程:", getattr(msg, "reasoning_content"))
print(f"Token 消耗: {response.usage.total_tokens}")

2. 开启思考模式（thinking: enabled）
回答: 小明最后还剩 **6个** 苹果。

计算过程：
1. 一开始有 5 个苹果；
2. 给了小红 2 个，剩下：5 - 2 = 3 个；
3. 又买了 3 个，变成：3 + 3 = 6 个。

所以最后小明还有 6 个苹果。
思考过程: 用户问：小明有5个苹果，给了小红2个，又买了3个，最后还剩几个？
计算过程：
初始：5个苹果
给了小红2个：5 - 2 = 3个
又买了3个：3 + 3 = 6个
最后还剩：6个。

需要用简单明了的语言回答。
Token 消耗: 184


3.推理深度对比

统一输入

In [5]:
# 推理深度对比
messages = [
    {"role": "user", "content": "请详细分析一下为什么天空是蓝色的。"},
]

print("\n=== 推理深度对比 ===")
print("问题:", messages[0]["content"])
print()


=== 推理深度对比 ===
问题: 请详细分析一下为什么天空是蓝色的。



low,medium,high

In [6]:
for effort in ["low", "medium", "high"]:
    print(f"{effort} 推理深度")
    response = client.chat.completions.create(
        model="hy3",
        messages=messages,
        extra_body={"reasoning_effort": effort},
    )

    msg = response.choices[0].message
    print("回答:", msg.content[:100], "..." if len(msg.content) > 100 else "")
    if hasattr(msg, "reasoning_content"):
        reasoning = getattr(msg, "reasoning_content")
        print("思考过程:", reasoning[:80], "..." if len(reasoning) > 80 else "")
    print(f"Token 消耗: {response.usage.total_tokens}")
    print()

low 推理深度
回答: 天空呈现蓝色，是太阳光与地球大气层相互作用的结果，其核心物理机制是**瑞利散射（Rayleigh scattering）**。下面从多个层面详细分析这一现象。

---

### 1. 太阳光的组成与 ...
思考过程: 我们需要详细分析为什么天空是蓝色的。这是一个经典的物理/大气散射问题。需要从瑞利散射、太阳光谱、大气成分、散射与波长的关系、人眼视觉等方面解释。还要提到为什么不 ...
Token 消耗: 1505

medium 推理深度
回答: 天空呈现蓝色，是太阳光与地球大气层相互作用的结果，核心机制是**瑞利散射（Rayleigh scattering）**。下面从多个层面详细分析这一现象。

---

### 1. 太阳光的组成
太阳发 ...
思考过程: 我们需要详细分析为什么天空是蓝色的。这是一个经典的科学问题，涉及瑞利散射、大气成分、太阳光组成等。用户要求“详细分析”，所以需要从基本原理讲起，包括：
1. 太 ...
Token 消耗: 1649

high 推理深度
回答: 天空之所以呈现蓝色，是物理学中一个非常经典且美妙的现象。它的核心原因在于**太阳光与地球大气层之间的相互作用**，具体是由一种被称为**“瑞利散射”（Rayleigh Scattering）**的物理 ...
思考过程: 用户要求详细分析为什么天空是蓝色的。
这是一个经典的物理/大气科学问题，主要涉及瑞利散射（Rayleigh scattering）、太阳光的组成、大气成分等。
 ...
Token 消耗: 3022

